# CEP: Corrected Efficient Market Hypothesis

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [2]:
import pandas as pd
from statsmodels.formula import api as smf
from data_helpers.data_loaders import PandasDataLoader 
from data_helpers.data_prep import dynamic_treatments
import numpy as np
from tqdm.auto import tqdm


## Load and filter dataset

In [3]:
model_name = 'cemh'
sample_splits_df = pd.read_feather('../../../data/preprocessed/train_test_split.ft')
time_aggregated_dataset = pd.read_feather('../../../data/preprocessed/time_aggregate_dataset.ft')
time_aggregated_dataset = time_aggregated_dataset[~time_aggregated_dataset.treatment.isin(dynamic_treatments)]
time_aggregated_dataset = time_aggregated_dataset.query('round <= 5 and time <= 120')

## Keep relevant columns and prepare train-test loader

In [4]:
key_columns = ['treatment', 'game', 'round', 'time', 'n_unique_deals_round']
rounds = range(1,5)
n_deal_prices = range(0,6)
pdl = PandasDataLoader(sample_splits_df, time_aggregated_dataset)
target_col = 'ce_round'

## Fit and evaluate models

In [5]:
np.random.seed(1)
all_results = []
regression_res = []
for i in tqdm(range(pdl.max_samples)):
    # Get sample split
    train_df, test_df = pdl.get_sample_split_dataset(i)

    # Prepare linear regression patsy formula. This model indicates a grouped regression on price rule and feedback setting without the intercept.
    
    formula = target_col+'~price_rule:feedback_setting:realized_price-1'

    # for each round
    for rd in rounds:
        # for each number of deal prices
        for n_deal_price in n_deal_prices:
            # fit a model based on predermined formula and keep all data points of the training set that have less number of observed deal prices.
            model = smf.rlm(formula, data=train_df.query('n_unique_deals_round ==  ' + str(n_deal_price)))
            best_model = model.fit()

            # fetch the test set
            test_query = 'round == ' + str(rd) + ' and n_unique_deals_round ==  ' + str(n_deal_price)
            sub_test_set = test_df.query(test_query)            

            # predict on the test set
            prediction = best_model.predict(sub_test_set)
            test_targets = sub_test_set[target_col]
            
            # append results to dataframe of results
            result_test_df = sub_test_set[key_columns].copy()
            result_test_df.loc[:, 'sample_id'] = i
            result_test_df.loc[:, 'ce_ape'] = (np.abs(prediction - test_targets)/test_targets)
            all_results.append(result_test_df)

            # append parameters to dataframe of parameters.
            a = (best_model.summary2().tables[1]).stack().to_frame().T.swaplevel(-2, -1, axis=1)
            a['sample_id'] = i
            a['round'] = rd
            a['n_deal_prices'] = n_deal_price
            regression_res.append(a)

  0%|          | 0/50 [00:00<?, ?it/s]

## Combine results

In [6]:
all_results_df = pd.concat(all_results, ignore_index = True)
all_results_df['model'] = model_name
regression_data_df = pd.concat(regression_res, axis=0, ignore_index = True)
regression_data_df['model'] = model_name

## Persist Results

In [7]:
all_results_df.to_feather('../../../data/results/ce_price/'+model_name+'.ft')
regression_data_df.to_feather('../../../data/results/ce_price/'+model_name+'_data.ft')